# 🎬 Manga Recap AI Studio (Google Colab T4 + Gradio UI)
> **Tự động hóa sản xuất Video Tóm tắt / Review Truyện tranh Kịch tính (Quán Khuya Style)**  
> Mặc định môi trường: **Python 3 + GPU NVIDIA Tesla T4 (16GB VRAM)**.  
> Tích hợp **Giao diện Gradio Web UI** trực quan, hỗ trợ xem trước và tương tác dễ dàng.

### ⚡ Bước 1: Kiểm tra GPU T4 & Kết nối Google Drive (Tùy chọn)
- **Lưu ý:** Bạn có thể kết nối Google Drive để lưu trữ video thành phẩm vĩnh viễn.
- **Nếu KHÔNG mount Google Drive:** Hệ thống sẽ tự động lưu video an toàn vào thư mục `/content/workspace` trên Colab.

In [1]:
# 1. Kiểm tra cấu hình GPU Tesla T4
!nvidia-smi

# 2. Kết nối Google Drive (Nếu không mount, hệ thống tự động lưu vào /content/workspace)
import os
from pathlib import Path

USE_DRIVE = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = Path('/content/drive/MyDrive/MangaRecap')
    drive_dir.mkdir(parents=True, exist_ok=True)
    USE_DRIVE = True
    print("✅ Đã kết nối Google Drive! Kết quả sẽ tự động sao lưu vào /content/drive/MyDrive/MangaRecap")
except Exception as e:
    print("ℹ️ Chưa kết nối Google Drive. Toàn bộ file sẽ được lưu cục bộ tại /content/workspace trên Colab.")

Sat Oct  3 16:36:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 📦 Bước 2: Cài đặt Toàn Bộ Thư Viện Cần Thiết
Cài đặt Gradio, VieNeu-TTS, Faster-Whisper, Ultralytics, Google GenAI SDK...

In [6]:
!git clone https://github.com/tuna1710/REVIEWTRUYENTRANH.git
%cd REVIEWTRUYENTRANH

Cloning into 'REVIEWTRUYENTRANH'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 35 (delta 5), reused 35 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 38.08 KiB | 3.81 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/REVIEWTRUYENTRANH


In [7]:
# Cài đặt các thư viện trong requirements.txt
!pip install -q -r requirements.txt
print("✅ Đã cài đặt xong toàn bộ môi trường!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 941.3/941.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.9/334.9 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.

### 🌟 Bước 3: Khởi Chạy Giao Diện Gradio Web UI
Chạy lệnh bên dưới để mở giao diện web trực quan.  
Gradio sẽ cung cấp:  
1. **Giao diện hiển thị trực tiếp bên trong Colab.**  
2. **Đường link Public (`https://xxxx.gradio.live`)** để bạn có thể mở toàn màn hình trên trình duyệt hoặc điện thoại.

In [8]:
# Khởi chạy Gradio Web App
!python app.py

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
/content/REVIEWTRUYENTRANH/app.py:208: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, title="Manga Recap AI Studio") as demo:
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://a360f2c7792acd8925.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Tracebac

### 💻 [Tùy Chọn] Bước 4: Chạy Bằng Lệnh Terminal (CLI)
Nếu không muốn dùng giao diện Web, bạn có thể chạy trực tiếp bằng dòng lệnh:

In [9]:
!python main.py \
  --input "https://link-truyen-tranh-chapter-1" \
  --synopsis "Ngôi làng bí ẩn với những cái chết kỳ dị..." \
  --voice "NamMinh" \
  --capcut

🔥 MANGA RECAP AI STUDIO - AUTOMATED RECAP PIPELINE 🔥

--- [STEP 1/5] Ingesting Manga Chapter ---
[Downloader] Downloading chapter from: https://link-truyen-tranh-chapter-1
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/socket.py", line 983, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno -2] Name or service not known

The above exception was